# Portfolio formulation

This notebook formalizes the continuous MVO baseline and the discrete K-cardinality selection problem used by GA, SA, and QAOA.

In [1]:
import numpy as np
import pandas as pd

from portfolio_opt.data import build_default_price_data, calculate_returns, estimate_covariance, estimate_expected_returns
from portfolio_opt.portfolio import build_discrete_qubo, discrete_objective, exact_enumeration, solve_mvo

In [2]:
prices = build_default_price_data()
returns = calculate_returns(prices)
mu = estimate_expected_returns(returns, annualization=252)
sigma = estimate_covariance(returns, annualization=252)
mu.head()

AAPL     0.574522
MSFT     0.472719
AMZN     0.676325
GOOGL    0.750668
dtype: float64

## Continuous MVO baseline

maximize $\mu^T w - \lambda w^T \Sigma w$ subject to $\sum_i w_i = 1$ and $w_i \geq 0$.

In [3]:
weights = solve_mvo(mu.to_numpy(), sigma.to_numpy(), risk_aversion=1.0)
weights

array([7.59366021e-09, 6.48482739e-09, 5.06373495e-02, 9.49362636e-01])

## Discrete K-cardinality formulation

For binary selection variables $x_i \in \{0,1\}$, we use equal-weight portfolios over exactly $K$ selected assets. The objective is:

$$\max_{x} \left( \frac{1}{K} \mu^T x - \lambda \frac{1}{K^2} x^T \Sigma x \right)$$

subject to $\sum_i x_i = K$.

In [4]:
k = 2
qubo, linear, penalty, meta, decoder = build_discrete_qubo(mu.to_numpy(), sigma.to_numpy(), k, risk_aversion=1.0)
best = exact_enumeration(mu.to_numpy(), sigma.to_numpy(), k=k, risk_aversion=1.0)
print(best)
print('qubo.shape =', qubo.shape)
print('linear =', linear)

{'x': array([0, 0, 1, 1]), 'objective': 0.6846670961455383, 'k': 2}
qubo.shape = (4, 4)
linear = [-0.27431405 -0.22663497 -0.3215328  -0.36416891]
